# VLM-DENTAL — Evaluation (TPU): inventory and missing cohorts

> **TPU copy of `notebooks/VLM_Dental_Colab_Final_Evaluation.ipynb`.** Same evaluator, same profile tables; the defaults that must be picked by hand are preset:
> `EVAL_TARGET_PROFILES = "tpu"` (canonical letterbox views, static-shape generation, 24,576 new tokens: `docs/TPU_EVALUATION.md`), `EVAL_TRACK = "with_tools"`,
> `EVAL_STAGES = "multicohort_all"`, `EVAL_KINDS = ["zeroshot", "baseline", "sft", "grpo"]` (the GRPO cohorts are plain `off` and `rules_skills_index`), `EVAL_SELECTED_K = 4`, split `test`.
> TPU results are their own cohort (profile tag `tpu_bf16`); they are not mixed with GPU results.
>
> **Order (a failure stops the notebook):** checkout and install (TPU image's torch is kept), settings, **DEBUG block** (environment doctor + checkpoint presence on the Hub, then the read-only inventory), then the evaluations, then the summaries.
Author: Reza Nadimi <rezaxr14@gmail.com>

After the original zero-shot/SFT/GRPO/rules-skills/speculative workflows, inspect the existing local and Hub evidence and run only missing work. All notebooks share the same result identities. Training and frontier API jobs are never started here. Results go to the existing `Reza-Nadimi/vlm-dental-evaluations` repository under `evaluations/dentex/`; this notebook does not build the paper.

## 1. Evaluation checkout

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"
REPO_REF = "main"  # or the exact reviewed commit containing this patch
if not Path("dental_agent/harness/notebook_eval.py").is_file():
    parent = Path("/kaggle/working" if Path("/kaggle").exists() else "/content" if Path("/content").exists() else Path.cwd())
    work = parent / "VLM-DENTAL-EVALUATION"
    if not work.exists():
        if len(REPO_REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REPO_REF):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(work)], check=True)
            subprocess.run(["git", "-C", str(work), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
            subprocess.run(["git", "-C", str(work), "checkout", "--detach", "FETCH_HEAD"], check=True)
        else:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(work)], check=True)
    os.chdir(work)
assert Path("dental_agent/harness/notebook_eval.py").is_file(), "Use the checkout containing this patch"
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
print("Checkout:", Path.cwd())

## 2. Dependencies and existing credentials

In [ ]:
INSTALL_DEPENDENCIES = True  # install only in this isolated evaluation runtime
if INSTALL_DEPENDENCIES:
    # TPU runtime: keep the image's preinstalled torch / torch_xla. Shared packages only, then the repo without dependency resolution.
    for _args, _must in ((["uninstall", "-y", "torchao"], False),
                         (["install", "-q", "-U", "transformers>=5.0.0", "accelerate", "peft", "trl", "datasets", "huggingface_hub", "ultralytics", "python-dotenv", "qwen-vl-utils", "tabulate", "tpu-info", "openai", "google-genai", "anthropic", "pytest"], True),
                         (["install", "-q", "pandas", "scikit-learn", "matplotlib", "pyarrow", "PyYAML", "click", "tqdm", "langgraph"], True),
                         (["install", "-q", "--no-deps", "-e", "."], True)):
        subprocess.run([sys.executable, "-m", "pip", *_args], check=_must)
from dotenv import load_dotenv
load_dotenv()
for key in ("HF_TOKEN", "HF_ARTIFACT_REPO", "DENTEX_IMAGES_REPO"):
    if os.environ.get(key): continue
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(key)
        elif IS_COLAB:
            from google.colab import userdata
            value = userdata.get(key)
        else: value = None
        if value: os.environ[key] = value
    except Exception: pass
print("Artifacts:", os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models"))

## 3. Profiles and cohorts
Set `EVAL_TARGET_PROFILES` to one profile, a list, or `"all"`. This selects checkpoint cohorts, not a claim that the current hardware supports every numerical runtime. Native BF16 and serving requirements are checked before weights load. Unsupported profiles remain listed as pending.

`EVAL_SELECTED_K` is fixed on dev before final test. Rules/skills and speculative comparisons always use the fixed dev split. The primary test cohort contains all 50 official images, including healthy cases. Zero-shot retains its own prompt and is never substituted for the unified no-tools baseline.

On a TPU (v5e-8) the notebook selects the `tpu` profile automatically: canonical black-bar views only (original-size cells are listed as unsupported), 32k/48k static-shape generation, results labelled `tpu_bf16`. A prompt that does not fit the static cache is recorded as a refused episode (no answer, flagged and counted), never truncated. Speculative decoding stays pending on TPU.

In [ ]:
from dental_agent.harness.notebook_eval import NotebookEvaluations
import json
# Profiles are visible here. Aliases: rtx4090 -> rtx4090_qlora; rtxpro6000_48gb -> workstation48_lora.
EVAL_PROFILES = {
    "dual_t4": {
        "tag": "dual_t4_fp16",
        "precision": "qlora",
        "compute": "fp16",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    },
    "rtx4090_qlora": {
        "tag": "rtx4090_bf16",
        "precision": "qlora",
        "compute": "bf16",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "workstation48_lora": {
        "tag": "workstation48_lora_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "quadro_rtx8000": {
        "tag": "quadro_rtx8000_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False,
        "serving_dtype": "float16"
    },
    "tpu": {
        "tag": "tpu_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "off",
        "loader": "standard",
        "threads": 0,
        "letterbox": True
    },
    "dual_t4_fp16": {
        "tag": "dual_t4_unquantized_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    }
}
# Existing result repository; checkpoints still come from HF_ARTIFACT_REPO.
EVAL_RESULTS_REPO = "Reza-Nadimi/vlm-dental-evaluations"
EVAL_RESULTS_REPO_TYPE = None  # Detect the existing model or dataset repository; never create one.
EVAL_TRACK = "with_tools"  # TPU notebook default (with_tools, no_tools, or all)
EVAL_GRPO_HARNESS_MODES = ["off", "rules_skills_index"]  # Separate trained GRPO cohorts
SPEC_HARNESS_MODE = "rules_skills_index"
SPEC_TARGET_KIND = "grpo"
SPEC_DRAFT_VARIANT = "original"  # original or finetuned; exact merged targets belong in SPEC_CHECKPOINTS
print(json.dumps(EVAL_PROFILES, indent=2))
# TPU is detected without starting XLA (the evaluation child process needs the TPU). On a TPU the "tpu" profile is
# selected: canonical letterbox views only, static-shape generation (docs/TPU_EVALUATION.md). Override below if needed.
from dental_agent.model.tpu_inference import tpu_runtime_present
IS_TPU = tpu_runtime_present()
EVAL_TARGET_PROFILES = "tpu"  # TPU notebook default (explicit; any key above stays selectable)
if not IS_TPU:
    raise RuntimeError("This notebook is the TPU copy (EVAL_TARGET_PROFILES = 'tpu') but no TPU runtime was detected; nothing switches automatically. Use notebooks/VLM_Dental_Colab_Final_Evaluation.ipynb for a GPU.")

EVAL_KINDS = ["zeroshot", "baseline", "sft", "grpo"]   # TPU run plan: speculative decoding is TPU_Speculative_Decoding.ipynb, rules/skills ablations come later
EVAL_STAGES = "multicohort_all"   # the stage trained by TPU_SFT / TPU_GRPO ("all" discovers every uploaded stage)
EVAL_SELECTED_K = 4
EVAL_SPLIT = "test"
EVAL_SIZES = ["canonical"]  # TPU: canonical letterbox views only
DATA_DIR = "data"
DEV_SPLIT = "data/splits/dentex_dev.json"
MODEL_ID = None
from dental_agent.harness.notebook_eval import generation_budget
MAX_NEW_TOKENS = generation_budget("tpu")  # 24576 on TPU (profile-defined in notebook_eval; every GPU profile stays 16384)
MAX_MODEL_LEN = 32768  # total context capacity handed to the evaluator
SPEC_CHECKPOINTS = {}  # {profile: {exact source adapter prefix: audited merged target}}
VLLM_EXECUTABLE = "vllm"  # isolated serving environment executable
SERVING_GPU_INDICES = "0"
SERVING_PORT = 8910
RUN_MISSING_EVALUATIONS = True  # False: inventory/restore only, no inference

## 3a. DEBUG FIRST: verbose environment doctor and checkpoint presence

TPU runtime, packages, `tpu-info`, host RAM and disk, Hub access, and whether the SFT / GRPO-off / GRPO-rules+skills checkpoints for the selected stage and K exist. Missing GRPO checkpoints are warnings (the evaluator lists them as `missing_checkpoint`); any FAIL stops the notebook.

In [ ]:
# =========================================================================
# DEBUG: VERBOSE ENVIRONMENT DOCTOR + CHECKPOINT PRESENCE. Runs before any evaluation; any FAIL raises. The read-only inventory follows.
# =========================================================================
NB_TAG = "final_evaluation"
NOTEBOOK_REPORT_DIR = Path("/kaggle/working/tpu_report/notebook_final_evaluation") if Path("/kaggle").exists() else Path("data/tpu_notebook_report")
REPORT_HUB_REPO = os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")
_TAG = EVAL_PROFILES["tpu"]["tag"]
REPORT_SUBFOLDER = f"tpu_reports/final_evaluation/{_TAG}/notebook"
HF_FILES = []

import importlib.metadata as _md
import json, platform, shutil, subprocess, sys, time, traceback
from datetime import datetime, timezone
from pathlib import Path
from tabulate import tabulate

NOTEBOOK_REPORT_DIR = Path(globals().get("NOTEBOOK_REPORT_DIR") or "data/tpu_notebook_report")
NOTEBOOK_REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_HUB_REPO = globals().get("REPORT_HUB_REPO") or os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")
DOCTOR = []

def record(status, name, detail=""):
    DOCTOR.append({"status": status, "check": name, "detail": str(detail)})
    print(f"[{status:<4}] {name}: {detail}", flush=True)

def guarded(name):
    def deco(fn):
        try:
            fn()
        except Exception as exc:
            record("FAIL", name, f"{type(exc).__name__}: {exc}")
            traceback.print_exc()
        return fn
    return deco

def push_notebook_report(reason):
    """Best effort: copy this notebook's own logs to the Hub run-report folder (an existing repo; never created)."""
    if not (REPORT_HUB_REPO and os.environ.get("HF_TOKEN")):
        print(f"[REPORT] kept local only ({reason}): {NOTEBOOK_REPORT_DIR}", flush=True)
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=os.environ["HF_TOKEN"])
        sent = []
        for p in sorted(NOTEBOOK_REPORT_DIR.glob("*")):
            if p.is_file() and p.suffix in (".json", ".jsonl", ".txt", ".log") and 0 < p.stat().st_size < 50_000_000:
                api.upload_file(path_or_fileobj=str(p), repo_id=REPORT_HUB_REPO, path_in_repo=f"{REPORT_SUBFOLDER}/{p.name}")
                sent.append(p.name)
        print(f"[REPORT] pushed {sent} to {REPORT_HUB_REPO}/{REPORT_SUBFOLDER} ({reason})", flush=True)
    except Exception as exc:
        print(f"[REPORT WARNING] Hub push failed ({reason}): {type(exc).__name__}: {exc}", flush=True)

def run_streamed(argv, env, log_path, label):
    """Run argv, echo every line (stdout+stderr merged) to the notebook and to log_path. Returns the exit code."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.time()
    with open(log_path, "w", encoding="utf-8") as log, subprocess.Popen(
        [str(a) for a in argv], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    ) as proc:
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = proc.wait()
        except BaseException:
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
            raise
    print(f"[{label}] exit code {code} after {time.time() - started:.0f}s; full output saved to {log_path}", flush=True)
    return code

def fail_with_tail(label, code, log_path, lines=80):
    tail = Path(log_path).read_text(encoding="utf-8", errors="replace").splitlines()[-lines:]
    print(f"\n[{label} FAILED] exit code {code}. Last {len(tail)} lines:\n" + "\n".join(tail), flush=True)
    push_notebook_report(f"{label} failed")
    raise RuntimeError(f"[{label} FAILED] exit code {code}. Evidence: {NOTEBOOK_REPORT_DIR} and the Hub folder {REPORT_SUBFOLDER}.")

print("=" * 78); print(f"DEBUG: ENVIRONMENT DOCTOR ({NB_TAG})  " + datetime.now(timezone.utc).isoformat(timespec="seconds")); print("=" * 78)

@guarded("runtime")
def _():
    record("INFO", "python / platform", f"{sys.version.split()[0]} / {platform.platform()}")
    record("INFO", "cwd", Path.cwd())
    for k in sorted(os.environ):
        if any(t in k for t in ("PJRT", "TPU", "XLA", "LIBTPU", "KAGGLE_KERNEL")) and not any(s in k for s in ("KEY", "TOKEN", "SECRET")):
            record("INFO", f"env {k}", os.environ[k][:200])

@guarded("devices")
def _():
    accel = sorted(p.name for p in Path("/dev").glob("accel*"))
    vfio = sorted(p.name for p in Path("/dev/vfio").glob("*")) if Path("/dev/vfio").exists() else []
    record("PASS" if (accel or vfio) else "WARN", "TPU device nodes", f"/dev/accel*={accel} /dev/vfio/*={vfio}")
    try:
        from dental_agent.model.tpu_inference import tpu_runtime_present
        record("PASS" if tpu_runtime_present() else "FAIL", "tpu_runtime_present (no XLA init)", "the evaluation child process needs the TPU free")
    except Exception as exc:
        record("WARN", "tpu_runtime_present", f"{type(exc).__name__}: {exc}")

@guarded("packages")
def _():
    must = {"torch", "torch_xla", "transformers", "peft"}
    for pkg in ("torch", "torch_xla", "libtpu", "transformers", "peft", "trl", "accelerate", "datasets", "huggingface_hub", "qwen-vl-utils", "safetensors", "openai", "google-genai", "anthropic", "tpu-info"):
        try:
            ver = _md.version(pkg)
            bad = pkg == "transformers" and int(ver.split(".")[0]) < 5
            record("FAIL" if bad else "INFO", f"pkg {pkg}", ver + ("  (Qwen3.5 needs transformers>=5.0.0)" if bad else ""))
        except _md.PackageNotFoundError:
            record("FAIL" if pkg in must else "WARN", f"pkg {pkg}", "not installed")

@guarded("tpu-info")
def _():
    try:
        r = subprocess.run(["tpu-info"], capture_output=True, text=True, timeout=90)
    except FileNotFoundError:
        record("WARN", "tpu-info", "CLI not found"); return
    print(r.stdout); print(r.stderr)
    record("PASS" if r.returncode == 0 else "WARN", "tpu-info", f"exit {r.returncode}")

@guarded("host resources")
def _():
    mem = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo")}
    record("INFO", "host RAM", f"total {mem['MemTotal'] / 2**20:.1f} GiB, available {mem['MemAvailable'] / 2**20:.1f} GiB")
    for p in (".", "/kaggle/working", "/tmp"):
        if Path(p).exists():
            du = shutil.disk_usage(p)
            record("INFO", f"disk {p}", f"free {du.free / 2**30:.1f} GiB of {du.total / 2**30:.1f} GiB")

@guarded("hugging face")
def _():
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    who = api.whoami()
    record("PASS", "hf whoami", f"{who.get('name')} role={who.get('auth', {}).get('accessToken', {}).get('role')}")
    HF_FILES.clear()
    HF_FILES.extend(api.list_repo_files(REPORT_HUB_REPO))
    record("PASS", "hf artifact repo", f"{REPORT_HUB_REPO} reachable, {len(HF_FILES)} files")

def has_prefix(prefix):
    return any(f.startswith(prefix) for f in HF_FILES)

def finish_doctor():
    print("\n" + tabulate([[r["status"], r["check"], r["detail"][:110]] for r in DOCTOR], headers=["", "check", "detail"], tablefmt="simple"))
    (NOTEBOOK_REPORT_DIR / "doctor.json").write_text(json.dumps({"time": datetime.now(timezone.utc).isoformat(), "notebook": NB_TAG, "checks": DOCTOR}, indent=2))
    fails = [r for r in DOCTOR if r["status"] == "FAIL"]
    if fails:
        push_notebook_report("doctor FAIL")
        raise RuntimeError("[DOCTOR FAILED] " + "; ".join(f"{r['check']}: {r['detail'][:120]}" for r in fails))
    print(f"\n[DOCTOR OK] {sum(r['status'] == 'PASS' for r in DOCTOR)} PASS, {sum(r['status'] == 'WARN' for r in DOCTOR)} WARN, 0 FAIL")

@guarded("selection")
def _():
    record("INFO", "selection", f"profiles={EVAL_TARGET_PROFILES} track={EVAL_TRACK} stages={EVAL_STAGES} kinds={EVAL_KINDS} K={EVAL_SELECTED_K} split={EVAL_SPLIT} sizes={EVAL_SIZES} max_new_tokens={MAX_NEW_TOKENS}")
    record("PASS" if (EVAL_TARGET_PROFILES == "tpu" and IS_TPU) else "FAIL", "tpu selection", f"EVAL_TARGET_PROFILES={EVAL_TARGET_PROFILES} IS_TPU={IS_TPU}")

@guarded("checkpoints on the hub")
def _():
    from dental_agent.training.experiment_profiles import grpo_name
    precision = EVAL_PROFILES["tpu"]["precision"]
    stage = EVAL_STAGES
    sft = f"sft/qwen3_5_9b_sft_{EVAL_TRACK}_{stage}_{precision}__{_TAG}/"
    record("PASS" if has_prefix(sft + "adapter_config.json") else "WARN", "SFT adapter", sft)
    for mode in EVAL_GRPO_HARNESS_MODES:
        name = grpo_name(EVAL_TRACK, EVAL_SELECTED_K, stage, precision, _TAG, harness_mode=mode)
        prefix = f"grpo/{name}/grpo-{EVAL_TRACK}-final/"
        record("PASS" if has_prefix(prefix) else "WARN", f"GRPO {mode}", prefix + ("" if has_prefix(prefix) else "  (missing: that cohort will be listed as missing_checkpoint, not evaluated)"))

@guarded("images")
def _():
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    found = next((c for c in (Path(DATA_DIR) / "dentex" / "images", Path(DATA_DIR) / "dentex" / "train_images", Path(DATA_DIR) / "images") if c.is_dir()), None)
    n = sum(1 for _r, _d, fs in os.walk(found, followlinks=True) for f in fs if Path(f).suffix.lower() in exts) if found else 0
    record("INFO", "local DENTEX images", f"{found} -> {n} files (the evaluator fetches missing images from DENTEX_IMAGES_REPO)")

finish_doctor()

## 4. List available checkpoints and result bundles

In [ ]:
final_evaluations = NotebookEvaluations(grpo_harness_modes=EVAL_GRPO_HARNESS_MODES, speculative_harness_mode=SPEC_HARNESS_MODE, speculative_target_kind=SPEC_TARGET_KIND, draft_variant=SPEC_DRAFT_VARIANT, track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES, kinds=EVAL_KINDS,
    stages=EVAL_STAGES, selected_k=EVAL_SELECTED_K, data_dir=DATA_DIR, dev_split=DEV_SPLIT,
    model_id=MODEL_ID, split=EVAL_SPLIT, sizes=EVAL_SIZES, spec_checkpoints=SPEC_CHECKPOINTS,
    vllm_executable=VLLM_EXECUTABLE, gpu_indices=SERVING_GPU_INDICES, port=SERVING_PORT, serving_context_length=MAX_MODEL_LEN)
AVAILABLE_COHORTS = final_evaluations.list()

## 5. Restore completed evaluations and run missing work
Order: zero-shot → baseline without/with tools → SFT controls → GRPO controls → rules/skills → speculative decoding. Each source checkpoint and profile has its own content identity. Saved episodes resume; complete cohorts skip inference. Failed equivalence gates stay visible and suppress speedup claims. Missing checkpoints or audited merged targets are reported explicitly.

Progress is saved every 60 seconds and after handled interruption/completion. A hard runtime shutdown may lose the unsaved interval. No result bundle contains base weights, optimizer state, credentials or radiograph files.

In [ ]:
FINAL_EVAL_STATUS = final_evaluations.run(execute=RUN_MISSING_EVALUATIONS)
print(json.dumps(FINAL_EVAL_STATUS, indent=2))(NOTEBOOK_REPORT_DIR / "eval_status.json").write_text(json.dumps(FINAL_EVAL_STATUS, indent=2, default=str))
push_notebook_report("evaluation run finished")


## 6. Inspect measured metrics and evidence locations

In [ ]:
from dental_agent.harness.notebook_eval import summarize_inventory
FINAL_SUMMARY = summarize_inventory(FINAL_EVAL_STATUS)
print(json.dumps(FINAL_SUMMARY, indent=2))
print("Local inventory:", final_evaluations.output/"inventory.json")
print("Hub:", "https://huggingface.co/"+("datasets/" if final_evaluations.results_repo_type == "dataset" else "")+final_evaluations.results_repo+"/tree/main/evaluations")

## 7. Optional dev-only K sweep and tool/grounding probes
These preserve the development experiments without changing the final test selection. Each probe has its own identity and Hub bundle.


In [ ]:
RUN_DEV_K_SWEEP = False
RUN_DEV_PROBES = False
DEV_ABLATIONS = [dict(remove_tools=["nudge_crop"], locate_shift_frac=0.0, locate_scale=1.0, seed=42),
                 dict(remove_tools=[], locate_shift_frac=0.2, locate_scale=1.0, seed=42)]
if RUN_DEV_K_SWEEP:
    dev_sweep = NotebookEvaluations(grpo_harness_modes=EVAL_GRPO_HARNESS_MODES, speculative_harness_mode=SPEC_HARNESS_MODE, speculative_target_kind=SPEC_TARGET_KIND, draft_variant=SPEC_DRAFT_VARIANT, track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES, kinds=["sft", "grpo"],
        stages=EVAL_STAGES, selected_k="all", data_dir=DATA_DIR, dev_split=DEV_SPLIT,
        model_id=MODEL_ID, split="dev", sizes=EVAL_SIZES)
    dev_sweep.list()
    DEV_K_STATUS = dev_sweep.run()
if RUN_DEV_PROBES:
    DEV_PROBE_STATUS = []
    for ablation in DEV_ABLATIONS:
        probe = NotebookEvaluations(grpo_harness_modes=EVAL_GRPO_HARNESS_MODES, speculative_harness_mode=SPEC_HARNESS_MODE, speculative_target_kind=SPEC_TARGET_KIND, draft_variant=SPEC_DRAFT_VARIANT, track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES, kinds=["baseline", "sft", "grpo"],
            stages=EVAL_STAGES, selected_k=EVAL_SELECTED_K, data_dir=DATA_DIR, dev_split=DEV_SPLIT,
            model_id=MODEL_ID, split="dev", ablation=ablation)
        DEV_PROBE_STATUS.extend(probe.run())


## 8. Save paired statistics beside the evaluations
This writes JSON and Markdown to the existing artifact repository. No paper assets are generated.


In [ ]:
from dental_agent.harness.notebook_eval import publish_paired_summaries
PAIRED_SUMMARIES = publish_paired_summaries(FINAL_EVAL_STATUS)
print(json.dumps(PAIRED_SUMMARIES, indent=2))


## 9. Tables and Hub files per split and harness mode (separate folders, separate files)

The saved cohorts of section 8 stay exactly where they are. `scripts/flatten_results.py` copies them into `data/evaluations_flat_k<K>/` (one root per `EVAL_SELECTED_K`) with one folder per split and harness mode: `test/` and `dev/` (GRPO trained without the harness), `test__harness_<mode>/` and `dev__harness_<mode>/` (the GRPO trained with it, plus the baseline and SFT controls it is compared with), `rules_skills/` and `speculative/` (the result documents of those cohorts). `scripts/aggregate_eval.py` writes the tables of each test/dev folder. Optionally every folder is uploaded as separate files under `aggregates/dentex/<area>/<tag>/<stage>/k<K>/run_<id>/` of the existing results repository: a changed table goes to a new `run_<id>` folder and nothing already on the Hub is overwritten. The per-cohort bundles and the paired summaries of section 8 are not touched. Run it once for every K you evaluated, changing only `EVAL_SELECTED_K`.

In [ ]:
import subprocess, sys
from pathlib import Path
RUN_AGGREGATION = False        # after the evaluations above: copy results into separate folders and write the tables (no model is loaded)
PUBLISH_AGGREGATES = False     # then upload each folder as separate files under aggregates/ of EVAL_RESULTS_REPO (nothing is overwritten)
AGG_PROFILE = "tpu"
AGG_STAGE = "multicohort_all"
FLAT_DIR = f"data/evaluations_flat_k{EVAL_SELECTED_K}"  # one folder root per K
AGG_TAG = EVAL_PROFILES[AGG_PROFILE]["tag"]

def _run(*args):
    print("$ python", *args, flush=True)
    subprocess.run([sys.executable, *args], check=True)

def _areas():
    return sorted(p.parent.name for p in Path(FLAT_DIR).glob("*/copied_files.json") if not p.parent.name.startswith("k_sweep"))

if RUN_AGGREGATION:
    _run("scripts/flatten_results.py", "--k", str(EVAL_SELECTED_K), "--tag", AGG_TAG, "--stage", AGG_STAGE, "--out", FLAT_DIR)
    for _area in _areas():
        if _area.split("__")[0] in ("test", "dev"):
            _run("scripts/aggregate_eval.py", "--output-dir", f"{FLAT_DIR}/{_area}", "--dataset", "dentex", "--split", _area.split("__")[0])
if PUBLISH_AGGREGATES:
    _run("scripts/publish_aggregates.py", "--areas", *_areas(), "--k", str(EVAL_SELECTED_K), "--tag", AGG_TAG, "--stage", AGG_STAGE,
         "--repo", EVAL_RESULTS_REPO, *(["--repo-type", EVAL_RESULTS_REPO_TYPE] if EVAL_RESULTS_REPO_TYPE else []))